In [ ]:
import csv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup
import time  

# 設定 Chrome WebDriver 的無頭模式
chrome_options = Options()
chrome_options.add_argument("--headless")  # 無頭模式（可選）

# 初始化 WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# 打開目標網頁
url = 'https://service.mof.gov.tw/public/Data/statistic/d3.js/demo/tax02/index.html'
driver.get(url)

try:
    # 等待頁面中的下拉選單載入
    wait = WebDriverWait(driver, 3)
    year_slider = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="range"]')))

    # 設定年度範圍
    for desired_year in range(98, 111):
        driver.execute_script("arguments[0].value = arguments[1];", year_slider, desired_year)

        # 手動觸發 change 事件，讓頁面更新年份並重新載入資料
        driver.execute_script("arguments[0].dispatchEvent(new Event('change'));", year_slider)
        
        # 查找下拉選單
        dropdown = wait.until(EC.presence_of_element_located((By.ID, 'variable')))
        select = Select(dropdown)

        # 遍歷下拉選單中的每個選項
        for option in select.options:
            # 取得選項的值和文字
            value = option.get_attribute("value")
            text = option.text

            # 選擇當前選項
            select.select_by_value(value)
            
            # 等待頁面載入（視情況調整時間）
            time.sleep(2)  # 暫停 2 秒，確保頁面載入完成
            
            # 點擊單選按鈕 "twn"
            radio_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'input[type="radio"][value="twn"]')))
            radio_button.click()

            # 等待頁面更新
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'td.sorttable_td')))

            # 取得更新後的頁面內容
            html_content = driver.page_source
            
            # 使用 BeautifulSoup 解析 HTML
            soup = BeautifulSoup(html_content, 'html.parser')

            # 查找所有具有 class "sorttable_td" 的 <td> 元素
            td_elements = soup.find_all('td', class_='sorttable_td')
            text_content = soup.find('text', class_='st_svg_title').get_text()

            # 創建一個列表來存儲擷取的行數據
            rows = []
            current_row = []

            # 遍歷每個 <td> 元素，並將它們兩兩組合成一行
            for td in td_elements:
                current_row.append(td.text)
                if len(current_row) == 2:  # 每兩項為一行
                    rows.append(current_row)
                    current_row = []  # 重置行

            # 生成 CSV 檔案名稱（根據選項的值命名）
            csv_filename = f"{desired_year}_{text_content}.csv"

            # 將擷取的數據寫入 CSV 檔案
            with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                # 寫入 CSV 檔案的標題行
                writer.writerow(['地區', f'{text_content}'])
                # 寫入所有行
                writer.writerows(rows)

            print(f"已儲存至 {csv_filename}")

finally:
    driver.quit()

保存到 98_申報戶數(戶).csv
保存到 98_綜合所得總額-平均數(萬元).csv


KeyboardInterrupt: 